# 15 - Long-Term Memory Across Conversations

## Purpose

This notebook demonstrates a LangChain DeepAgents tutorial episode. It walks through the key concepts and code patterns for building, configuring, and operating AI agents with the `deepagents` framework.

## What You'll Learn

- How deep agents differ from regular LangChain agents
- The built-in tool suite every agent receives automatically
- How to configure models, backends, permissions, and memory
- Practical patterns for tools, planning, memory, and handoffs

## Tutorial Series Context

This is Episode 15-long_term_memory of the LangChain DeepAgents YouTube tutorial series. Each episode focuses on a specific aspect of agent development, from basic introduction to advanced patterns like subagents, memory, permissions, and skills.

## How to Use This Notebook

1. **Run cells top-to-bottom**: The code is designed to execute in order, with each section building on the previous one.
2. **Observe tool calls**: Watch for `tool_calls` in message outputs - this shows which agents/tools are being used.
3. **Modify and experiment**: The code is intentionally kept identical to the original Python scripts. Feel free to modify parameters, models, or prompts to see how behavior changes.
4. **Refer to comments**: Each code section has accompanying markdown explanations of what the code does and why.

## Key Concepts Covered in This Episode

- StoreBackend persists memory across conversations
- InMemoryStore for dev; Postgres for production
- namespace receives Runtime, scopes per user in deployment
- /memories/ path is persistent, outside conversation history
- save by writing to MEMORY_FILE in system prompt
- Different thread_id = no shared message history
- Anything agent knows from memory file, not conversation


# Overview: 15-long_term_memory.ipynb

This notebook converts the Python tutorial `15-long_term_memory.ipynb` into an interactive Jupyter Notebook format, preserving all code examples and explanations exactly as they appear in the original tutorial scripts.

## Notebook Structure

Each logical section of the original Python script has been split into its own code cell. Each code cell is preceded by a markdown cell containing:

- **A short heading** describing what that section does
- **1–3 sentences** explaining the purpose and key concepts of that section

## How to Navigate

- **Run cells in order**: The tutorial was designed to be executed top-to-bottom. Each section builds on the previous one.
- **Observe tool calls**: Watch the output for `tool_calls` entries - this shows which built-in or tools the agent invokes.
- **Read the markdown comments**: Each code section has a preceding explanation of what the code does and why it's structured that way.
- **Experiment freely**: The original `.py` files are unchanged. Feel free to modify, extend, or experiment with the code in this notebook.

## Educational Value

This notebook serves as both a working tutorial replication and a learning resource. By running each cell and reading the accompanying explanations, you'll gain hands-on experience with:

- LangChain DeepAgents framework architecture
- Backend configuration (virtual vs real filesystem)
- Tool definition and registration
- Planning, memory, permissions, and context management
- Subagent delegation and structured output
- Streaming modes and human-in-the-loop patterns

## Tips for Getting the Most Out of This Notebook

1. **Don't skip the markdown cells**: The educational content between code sections is essential for understanding the "why" behind the code.
2. **Re-run after modifications**: After experimenting with changes, re-run all cells from the top to ensure everything still works together.
3. **Refer to the original Python scripts**: The `.py` files in the same directory are unchanged and can be referenced for the exact original code.
4. **Use rich output**: This notebook uses Rich formatting for colorful console output - pay attention to color-coded hints and status messages.

---

_Notebook automatically generated from `15-long_term_memory.ipynb`. Original Python scripts preserved unchanged._


## Section 2: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section sets up the imports and environment configurati
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
"""
15 - Long-Term Memory Across Conversations
==========================================
- Episode 12's checkpointer remembers ONE thread; a store remembers ALL of them
- Route /memories/ to a `StoreBackend` so files outlive the conversation
- `memory=[...]` loads those files into the system prompt on every run
Run:  python 15-long_term_memory.py
"""

## Section 4: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 1: Imports and setup
import os

from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, StateBackend, StoreBackend
from dotenv import load_dotenv
from langgraph.store.memory import InMemoryStore  # dev store; swap for Postgres
from rich import print

load_dotenv()
MODEL = os.getenv("DEEPAGENTS_MODEL", "ollama:qwen3.5:2b")


def text_of(message) -> str:
    """Normalize message content across providers (str vs content blocks)."""
    content = message.content
    if isinstance(content, str):
        return content
    return "".join(block.get("text", "") for block in content if isinstance(block, dict))

## Section 6: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements a store, and a namespace to file thi
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 2: A store, and a namespace to file things under
# `namespace` is a CALLABLE receiving the run's Runtime, so real deployments can
# scope memory per user, e.g. lambda rt: (rt.server_info.user.identity,).
# Locally there is no user, so we pin everything to one bucket.
store = InMemoryStore()
memory_backend = StoreBackend(store=store, namespace=lambda runtime: ("demo-user",))

## Section 8: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 3: Mount the store at /memories/
# Everything else stays virtual and disappears when the run ends — only the
# /memories/ path is persistent.
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": memory_backend},
)

MEMORY_FILE = "/memories/preferences.md"

agent = create_deep_agent(
    model=MODEL,
    backend=backend,
    memory=[MEMORY_FILE],  # loaded into the system prompt at the start of each run
    system_prompt=(
        "You are a personal assistant. When the user states a durable "
        f"preference, save it by writing to {MEMORY_FILE}. Reply in one sentence."
    ),
)


def say(text: str, thread_id: str) -> str:
    """Send one message on a named thread and return the reply."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config={"configurable": {"thread_id": thread_id}},
    )
    return text_of(result["messages"][-1]).strip()

## Section 10: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 4: Teach it something on thread one
first_turn = "I'm vegetarian - always remember that. Save it to your memory file."
print("[bold cyan]Conversation 1 (thread 'monday'):[/bold cyan]")
print(f"  [dim]user:[/dim]  {first_turn}")
print(f"  [green]agent:[/green] {say(first_turn, 'monday')}")

## Section 12: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for working with DeepAgents
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 5: Look at what actually got stored
# The file is now in the store, completely outside any conversation history.
print("\n[bold cyan]Contents of the store:[/bold cyan]")
for item in store.search(("demo-user",)):
    body = str(item.value.get("content", "")).strip().replace("\n", " ")
    print(f"  [yellow]{item.key}[/yellow]: {body[:100]}")

## Section 14: Key Concepts

This section of the tutorial focuses on implementing specific functionality for the LangChain DeepAgents framework. The code in this cell demonstrates core patterns for agent development.

**Code Overview:**
The Python code in this cell implements the featured capability for the DeepAgents framework. It shows the minimal pattern needed to add framework features to your agent.

**What You'll See:**

- Import statements and configuration needed for this feature
- The core function or agent setup pattern
- How this feature integrates with the rest of the DeepAgents system
- Key parameters and their effects on agent behavior

**Why This Pattern:**
This approach is designed to be reusable and extensible, allowing you to adapt it for your own projects while understanding the underlying framework concepts.

**Key Takeaways:**

- The code demonstrates the fundamental approach for This section implements start a brand new conversation and c
- Understanding the patterns here will help you implement similar features in your own agent projects
- The original Python script is preserved unchanged in the matching `.py` file

---

_Content adapted from LangChain DeepAgents tutorial episode structure._


In [ ]:
# %% Step 6: Start a BRAND NEW conversation and check recall
# Different thread_id = no shared message history (see ep. 12). Anything the
# agent knows here came from the memory file, not the conversation.
print("\n[bold cyan]Conversation 2 (thread 'friday' — no shared history):[/bold cyan]")
print("  [dim]user:[/dim]  Suggest a dinner for me.")
print(f"  [green]agent:[/green] {say('Suggest a dinner for me.', 'friday')}")

# %%